In [ ]:
"""
final_eval.py
Run on Colab after training + Optuna complete.
Confirms your true test set performance with tuned thresholds.
"""

import numpy as np
import pandas as pd
import torch
import json
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, hamming_loss

# ── Paths ───────────────────────────────────────────────────────────────────────
SAVE_PATH      = '/improved_model'
MODEL_PATH     = "improved_model/best_model"
TOKENIZER_PATH = "improved_model/best_tokenizer"
TEST_CSV       = "improved_model/test_set.csv"

CATEGORIES = [
    'Politics', 'Science and Technology', 'Business and Economy',
    'International Affairs', 'Education', 'Health', 'Sports', 'Entertainment'
]

# Exact Optuna values from your run
TUNED_THRESHOLDS = [
    0.6251138177843646,   # Politics
    0.7554300915494968,   # Science and Technology
    0.47825148961746794,  # Business and Economy
    0.7679008210439427,   # International Affairs
    0.5961489563153012,   # Education
    0.7344918680570672,   # Health
    0.6282793716255531,   # Sports
    0.7476509053277701,   # Entertainment
]

MAX_LEN    = 256
BATCH_SIZE = 32


class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row['title']).strip() + ' [SEP] ' + str(row['text']).strip()
        enc  = self.tokenizer(
            text, max_length=MAX_LEN, padding='max_length',
            truncation=True, return_tensors='pt', return_attention_mask=True
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(
                [float(row[c]) for c in CATEGORIES], dtype=torch.float32
            ),
        }


def evaluate_at_thresholds(probs, labels, thresholds, name):
    preds = np.array([
        (probs[:, i] > thresholds[i]).astype(int)
        for i in range(len(CATEGORIES))
    ]).T

    # Guarantee at least one label per article
    for i in range(len(preds)):
        if preds[i].sum() == 0:
            preds[i][np.argmax(probs[i])] = 1

    macro   = f1_score(labels, preds, average='macro',  zero_division=0)
    micro   = f1_score(labels, preds, average='micro',  zero_division=0)
    h_loss  = hamming_loss(labels, preds)
    exact   = np.mean(np.all(preds == labels, axis=1))
    per_lbl = f1_score(labels, preds, average=None, zero_division=0)

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Macro F1:     {macro:.4f}")
    print(f"  Micro F1:     {micro:.4f}")
    print(f"  Hamming Loss: {h_loss:.4f}")
    print(f"  Exact Match:  {exact:.4f}")
    print(f"\n  Per-label F1:")
    for cat, f1, t in zip(CATEGORIES, per_lbl, thresholds):
        flag = '  <- BELOW 0.80' if f1 < 0.80 else ''
        print(f"    {cat:<30} F1={f1:.4f}  t={t:.3f}{flag}")

    return macro, micro, h_loss, exact, per_lbl, preds


def main():
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    model     = AutoModelForSequenceClassification.from_pretrained(
                    MODEL_PATH).to(device)
    model.eval()

    test_df = pd.read_csv(TEST_CSV)
    print(f"Test set: {len(test_df)} articles")

    loader = DataLoader(
        TestDataset(test_df, tokenizer),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            all_logits.append(out.logits.cpu().numpy())
            all_labels.append(batch['labels'].numpy())

    logits = np.vstack(all_logits)
    labels = np.vstack(all_labels)
    probs  = 1 / (1 + np.exp(-logits))

    # Default vs tuned comparison
    evaluate_at_thresholds(
        probs, labels, [0.5] * len(CATEGORIES),
        "Default threshold 0.5 (baseline)"
    )

    macro, micro, h_loss, exact, per_lbl, preds = evaluate_at_thresholds(
        probs, labels, TUNED_THRESHOLDS,
        "Optuna-tuned thresholds (FINAL)"
    )

    # Save results
    results = {
        'macro_f1':       float(macro),
        'micro_f1':       float(micro),
        'hamming_loss':   float(h_loss),
        'exact_match':    float(exact),
        'per_label_f1':   dict(zip(CATEGORIES, per_lbl.tolist())),
        'thresholds':     dict(zip(CATEGORIES, TUNED_THRESHOLDS)),
        'n_test_samples': len(test_df),
    }

    # out_path = os.path.join(SAVE_PATH, 'final_test_results.json')
    # with open(out_path, 'w') as f:
    #     json.dump(results, f, indent=2)

    print(f"\nSaved -> {out_path}")
    print(f"\nFINAL Test Macro F1: {macro:.4f}")


if __name__ == '__main__':
    main()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Test set: 374 articles
